# 03 — Explore the Results

Loads the final VCF (variant call) file produced by Clair3 and shows a
simple, beginner-friendly summary: how many variants were found, what
types they are, and a preview table. Uses only `pandas` + the standard
library `gzip` module — no extra bioinformatics packages required.

## 1. Locate the output VCF

In [ ]:
import gzip, os

vcf_path = "results/variants/clair3_output/merge_output.vcf.gz"
assert os.path.exists(vcf_path), f"Can't find {vcf_path} — did the pipeline finish successfully?"
print("Found:", vcf_path)

## 2. Parse the VCF into a table

A VCF file has header lines starting with `#`, then one line per variant.
We skip the header and split each data line into columns.

In [ ]:
import pandas as pd

columns = ["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT", "SAMPLE"]
rows = []

with gzip.open(vcf_path, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        rows.append(fields[:len(columns)])

df = pd.DataFrame(rows, columns=columns)
df["POS"] = df["POS"].astype(int)
print(f"Total variants found: {len(df)}")
df.head(10)

## 3. Classify variants as SNP vs Indel

A SNP changes exactly one letter; an Indel changes the length of the sequence.

In [ ]:
def variant_type(row):
    ref, alt = row["REF"], row["ALT"].split(",")[0]
    if len(ref) == 1 and len(alt) == 1:
        return "SNP"
    return "Indel"

df["TYPE"] = df.apply(variant_type, axis=1)
df["TYPE"].value_counts()

## 4. Simple bar chart of variant types

In [ ]:
import matplotlib.pyplot as plt

df["TYPE"].value_counts().plot(kind="bar", title="Variant types called by Clair3")
plt.xlabel("Type")
plt.ylabel("Count")
plt.show()

## 5. Variants per chromosome/contig

In [ ]:
df["CHROM"].value_counts()

## Next steps

- Filter by `QUAL` to keep only high-confidence variants:
  `df[df["QUAL"].astype(float) > 20]`
- Export a filtered table: `df.to_csv("results/variants_summary.csv", index=False)`
- Load the VCF into IGV alongside `results/alignment/*.sorted.bam` for
  visual inspection of individual variants.